# 2. Distributions and risk factors

**Question.** Who reports heart disease: how does the rate change with age, sex, BMI, blood pressure,
smoking and diabetes, and do the measurements differ between people with and without it?

Counts and rates here describe the **unweighted** sample of respondents, with 95% Wilson intervals.
Section 2.5 shows what survey weights do to the same rates.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nhanes import nb, plots
from nhanes.features import CATEGORICAL, FEATURES, LABELS, NUMERIC, TARGET

ctx = nb.context()
df, R = ctx.df, ctx.R
pct = nb.pct
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 160)
print("mode:", "synthetic fixture (CI)" if ctx.fixture else "real NHANES data")
print(f"{len(df):,} adults, {int(df[TARGET].sum()):,} reporting heart disease")

mode: real NHANES data
22,512 adults, 1,637 reporting heart disease


## 2.1 Rate by risk factor

In [2]:
fig = plots.plot_rate_panels(df)
nb.show(fig)
d = plots.add_risk_factor_bins(df)
tab = (
    d.groupby("age_group", observed=True)[TARGET]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "rate", "sum": "cases", "count": "adults"})
)
display(tab)

<Figure size 1430x770 with 6 Axes>

,rate,cases,adults
age_group,,,
20s,0.003,11,3750
30s,0.012,46,3789
40s,0.025,92,3681
50s,0.061,223,3683
60s,0.120,466,3880
70-80,0.214,799,3729


## 2.2 Distributions by outcome

In [3]:
nb.show(plots.plot_distributions(df))
summary = (
    df.groupby(TARGET)[NUMERIC]
    .median()
    .T.rename(columns={0: "median, no heart disease", 1: "median, heart disease"})
)
display(summary)

<Figure size 1430x715 with 6 Axes>

heart_disease,"median, no heart disease","median, heart disease"
age_years,48.000,69.000
bmi,28.000,29.100
waist_cm,97.500,104.750
sbp_mean,121.333,129.333
dbp_mean,71.333,67.333
total_chol,188.000,167.000
hdl,51.000,46.000
hba1c,5.500,5.900
fasting_glucose,101.000,111.000
drinks_per_day,2.000,1.000


## 2.3 Age and sex together, and age and smoking together

Age is the strongest driver, so the other patterns must be read within an age group.


In [4]:
d = plots.add_risk_factor_bins(df)
by_sex = d.pivot_table(
    index="age_group", columns="sex", values=TARGET, aggfunc="mean", observed=True
)
by_smoke = d.pivot_table(
    index="age_group", columns="smoking_status", values=TARGET, aggfunc="mean", observed=True
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, t, title, cols in (
    (axes[0], by_sex, "By age and sex", {"female": "#E69F00", "male": "#0072B2"}),
    (
        axes[1],
        by_smoke,
        "By age and smoking",
        {"never": "#009E73", "former": "#D55E00", "current": "#CC79A7"},
    ),
):
    for c in t.columns:
        ax.plot(t.index.astype(str), t[c] * 100, marker="o", label=c, color=cols[c])
    ax.set(title=title, ylabel="% with heart disease")
    ax.legend(frameon=False)
fig.tight_layout()
nb.show(fig)
display(by_sex)

<Figure size 1210x440 with 2 Axes>

sex,female,male
age_group,,
20s,0.004,0.002
30s,0.012,0.012
40s,0.022,0.028
50s,0.050,0.072
60s,0.090,0.151
70-80,0.156,0.276


## 2.4 Cholesterol points the "wrong" way

Textbooks say higher cholesterol raises heart risk. Within an age band, compare the average total
cholesterol of people with and without a reported diagnosis.


In [5]:
chol = (
    df.groupby(["age_band", TARGET])
    .total_chol.mean()
    .unstack()
    .rename(columns={0: "no heart disease", 1: "heart disease"})
)
chol["difference (mg/dL)"] = chol["heart disease"] - chol["no heart disease"]
display(chol)
hdl = (
    df.groupby(["age_band", TARGET])
    .hdl.mean()
    .unstack()
    .rename(columns={0: "no heart disease", 1: "heart disease"})
)
display(hdl)

heart_disease,no heart disease,heart disease,difference (mg/dL)
age_band,,,
20-39,182.060,185.346,3.286
40-59,200.263,185.548,-14.715
60+,192.854,169.461,-23.393


heart_disease,no heart disease,heart disease
age_band,,
20-39,52.140,44.923
40-59,53.109,49.120
60+,55.564,49.679


## 2.5 Survey weights: do rates change?

In [6]:
ex = df[df.was_examined & (df.exam_weight > 0)]
w = ex.groupby("age_band").apply(
    lambda g: np.average(g[TARGET], weights=g.exam_weight), include_groups=False
)
u = ex.groupby("age_band")[TARGET].mean()
wt = pd.DataFrame({"unweighted": u, "weighted": w})
wt["ratio"] = wt.weighted / wt.unweighted
display(wt)
ws = R["weighted_sensitivity"]
nb.check(
    "overall weighted prevalence, test examined",
    float(
        np.average(
            df[(df.split == "test") & df.was_examined & (df.exam_weight > 0)][TARGET],
            weights=df[(df.split == "test") & df.was_examined & (df.exam_weight > 0)].exam_weight,
        )
    ),
    ws["weighted_prevalence"],
)

,unweighted,weighted,ratio
age_band,,,
20-39,0.008,0.007,0.909
40-59,0.043,0.039,0.906
60+,0.165,0.161,0.974


Check: overall weighted prevalence, test examined: notebook 0.0676787 vs results.json 0.0676787 (matches)


## 2.6 How related are the features to each other?

In [7]:
corr = df[NUMERIC].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.8, 5.6))
im = ax.imshow(corr, cmap="PuOr", vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC)), [LABELS[c] for c in NUMERIC], rotation=70, ha="right")
ax.set_yticks(range(len(NUMERIC)), [LABELS[c] for c in NUMERIC])
ax.grid(False)
fig.colorbar(im, label="Spearman correlation")
nb.show(fig)
pairs = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1))
    .stack()
    .sort_values(key=abs, ascending=False)
    .head(5)
)
display(pairs.rename("spearman").to_frame())

<Figure size 748x616 with 2 Axes>

spearman
bmi       waist_cm            0.907
hba1c     fasting_glucose     0.594
age_years hba1c               0.468
          sbp_mean            0.459
sbp_mean  dbp_mean            0.401

## What this means

In [8]:
age = tab["rate"]
sx = d.groupby("sex")[TARGET].mean()
sm = d.groupby("smoking_status")[TARGET].mean()
ages = d.groupby("smoking_status").age_years.mean()
dm = d.groupby("diabetes")[TARGET].mean()
c60 = chol.loc["60+", "difference (mg/dL)"] if "60+" in chol.index else float("nan")
nb.say(
    f"- The rate climbs from {pct(age.iloc[0])} in the youngest group to {pct(age.iloc[-1])} in the oldest "
    f"({age.iloc[-1] / max(age.iloc[0], 1e-9):.0f} times higher).\n"
    f"- Men report heart disease at {pct(sx['male'])} and women at {pct(sx['female'])}.\n"
    f"- By smoking: never {pct(sm['never'])}, former {pct(sm['former'])}, current {pct(sm['current'])}. "
    f"Mean age is {ages['never']:.1f} for never, {ages['former']:.1f} for former and {ages['current']:.1f} for current smokers, "
    f"so compare the groups within an age band (section 2.3).\n"
    f"- Adults told they have diabetes report heart disease at {pct(dm['yes'])} against {pct(dm['no'])} for those who are not.\n"
    f"- In the 60+ band, people with a reported diagnosis have average total cholesterol {c60:+.1f} mg/dL "
    f"relative to people without one."
)

- The rate climbs from 0.3% in the youngest group to 21.4% in the oldest (73 times higher).
- Men report heart disease at 9.1% and women at 5.5%.
- By smoking: never 4.9%, former 12.4%, current 8.0%. Mean age is 47.7 for never, 57.8 for former and 45.8 for current smokers, so compare the groups within an age band (section 2.3).
- Adults told they have diabetes report heart disease at 19.7% against 5.1% for those who are not.
- In the 60+ band, people with a reported diagnosis have average total cholesterol -23.4 mg/dL relative to people without one.

## What is weak

In [9]:
nb.say(
    "- These are **associations in a cross-section**. Smoking, BMI and blood pressure were measured at the same visit "
    "as the question, often years after any diagnosis, so the arrow of cause is unknown.\n"
    "- Cholesterol (and often blood pressure) may look better in people with heart disease because many take "
    "cholesterol-lowering or blood pressure drugs after diagnosis. The survey has those medication questions; "
    "this project deliberately does not use them as features because they are consequences of the diagnosis.\n"
    f"- Weighted and unweighted rates differ (the 60+ weighted-to-unweighted ratio is {wt.weighted.iloc[-1] / wt.unweighted.iloc[-1]:.2f}), "
    "so the unweighted figures describe the people surveyed, not the U.S. population.\n"
    f"- The most correlated pair of measurements is {pairs.index[0][0]} and {pairs.index[0][1]} (Spearman {pairs.iloc[0]:.2f}, see 2.6). "
    "Correlated inputs make individual coefficients unstable even when predictions are fine."
)

- These are **associations in a cross-section**. Smoking, BMI and blood pressure were measured at the same visit as the question, often years after any diagnosis, so the arrow of cause is unknown.
- Cholesterol (and often blood pressure) may look better in people with heart disease because many take cholesterol-lowering or blood pressure drugs after diagnosis. The survey has those medication questions; this project deliberately does not use them as features because they are consequences of the diagnosis.
- Weighted and unweighted rates differ (the 60+ weighted-to-unweighted ratio is 0.97), so the unweighted figures describe the people surveyed, not the U.S. population.
- The most correlated pair of measurements is bmi and waist_cm (Spearman 0.91, see 2.6). Correlated inputs make individual coefficients unstable even when predictions are fine.